# Demo: `ts_compress_to` with Timestamp Ordering

This notebook tests the compression table operator with `TIMESTAMP(6)` and
`TIMESTAMP(6) WITH TIME ZONE` ordering columns. It uses the same C runtime as
the sequence-number demo and overlays the compressed points on the original
timestamped series.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import teradatasql
from dotenv import load_dotenv
from IPython.display import Image, display

from td_ts_compressor import (
    CompressorSpec,
    TeradataTimeSeriesCompressor,
    analytical_points,
    visual_fidelity,
)


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "pyproject.toml").exists() else NOTEBOOK_DIR.parent
ENV_PATH = REPO_ROOT / ".env"
FIGURE_DIR = REPO_ROOT / "notebooks" / "assets"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_PATH)

required = ["TD_HOST", "TD_USERNAME", "TD_PASSWORD"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

OPERATOR_DATABASE = os.getenv("TD_OPERATOR_DATABASE") or os.getenv("TD_DATABASE")

print(f"Repository root: {REPO_ROOT}")
print("Compressor database:", OPERATOR_DATABASE or "current session database")


## 2. Connect to Teradata

In [ ]:
connect_args = {
    "host": os.environ["TD_HOST"],
    "user": os.environ["TD_USERNAME"],
    "password": os.environ["TD_PASSWORD"],
}
if os.getenv("TD_LOGMECH"):
    connect_args["logmech"] = os.environ["TD_LOGMECH"]
con = teradatasql.connect(**connect_args)
cur = con.cursor()
if os.getenv("TD_DATABASE"):
    cur.execute(f"DATABASE {os.environ['TD_DATABASE']}")
cur.execute("SELECT DATABASE")
print("Current database:", cur.fetchone()[0])

## 3. Generate Timestamp Datasets

In [ ]:
rng = np.random.default_rng(2027)
n_series = 12
n_rows_per_series = 240
seq = np.arange(n_rows_per_series, dtype=np.int64)
t = np.linspace(0, 1, n_rows_per_series)
base_ts = pd.Timestamp("2026-01-01 00:00:00")
offsets = ["+00:00", "+01:00", "-05:00", "+09:00"]

def make_signal(series_idx: int) -> tuple[str, np.ndarray]:
    family = series_idx % 4
    noise = rng.normal(0, 0.001 + 0.0002 * family, n_rows_per_series)
    base = 1.5 * series_idx
    if family == 0:
        label = "trend"
        y = base + 4.0 * t + noise
    elif family == 1:
        label = "periodic"
        y = base + 2.0 * np.sin(2 * np.pi * 6 * t) + noise
    elif family == 2:
        label = "peaks"
        y = base + noise
        for center in rng.choice(seq, size=4, replace=False):
            y += rng.uniform(2, 5) * np.exp(-0.5 * ((seq - center) / rng.uniform(3, 12)) ** 2)
    else:
        label = "step_noise"
        y = base + np.where(seq < 80, 0.0, np.where(seq < 160, 2.5, -1.0)) + noise
    return label, y.astype(float)

frames_ts, frames_tstz = [], []
for series_idx in range(n_series):
    pattern, y = make_signal(series_idx)
    timestamps = base_ts + pd.to_timedelta(seq * 15 + series_idx, unit="min")
    offset = offsets[series_idx % len(offsets)]
    frames_ts.append(pd.DataFrame({
        "series_id_1": pattern,
        "series_id_2": series_idx + 1,
        "ts": [ts.strftime("%Y-%m-%d %H:%M:%S.%f") for ts in timestamps],
        "y": y,
    }))
    frames_tstz.append(pd.DataFrame({
        "series_id_1": pattern,
        "series_id_2": series_idx + 1,
        "ts_tz": [f"{ts.strftime('%Y-%m-%d %H:%M:%S.%f')}{offset}" for ts in timestamps],
        "y": y,
    }))
ts_df = pd.concat(frames_ts, ignore_index=True)
tstz_df = pd.concat(frames_tstz, ignore_index=True)
print("TIMESTAMP rows:", len(ts_df))
print("TIMESTAMP WITH TIME ZONE rows:", len(tstz_df))
ts_df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for ax, series_id_2 in zip(axes.ravel(), [1, 2, 3, 4]):
    sample = ts_df[ts_df["series_id_2"] == series_id_2]
    ax.plot(pd.to_datetime(sample["ts"]), sample["y"], linewidth=1.0)
    ax.set_title(f"{sample['series_id_1'].iloc[0]} / series {series_id_2}")
    ax.grid(alpha=0.25)
fig.suptitle("Original timestamp-ordered series", y=1.02)
fig.tight_layout()
figure_path = FIGURE_DIR / "04_original_timestamp_series.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(figure_path)))

## 4. Load Permanent Teradata Tables

In [ ]:
def execute_ignore_error(sql: str) -> None:
    try:
        cur.execute(sql)
    except Exception as exc:
        if "3807" not in str(exc):
            print(f"Ignoring cleanup error: {exc}")

for table_name in ["input_timeseries_compress_ts", "input_timeseries_compress_tstz"]:
    execute_ignore_error(f"DROP TABLE {table_name}")

cur.execute("""
CREATE TABLE input_timeseries_compress_ts
(
    series_id_1 VARCHAR(128) CHARACTER SET LATIN,
    series_id_2 INTEGER,
    ts          TIMESTAMP(6),
    y           FLOAT
)
PRIMARY INDEX (series_id_1, series_id_2)
""")
cur.execute("""
CREATE TABLE input_timeseries_compress_tstz
(
    series_id_1 VARCHAR(128) CHARACTER SET LATIN,
    series_id_2 INTEGER,
    ts_tz       TIMESTAMP(6) WITH TIME ZONE,
    y           FLOAT
)
PRIMARY INDEX (series_id_1, series_id_2)
""")
cur.executemany(
    "INSERT INTO input_timeseries_compress_ts VALUES (?, ?, CAST(? AS TIMESTAMP(6)), ?)",
    list(ts_df.itertuples(index=False, name=None)),
)
cur.executemany(
    "INSERT INTO input_timeseries_compress_tstz VALUES (?, ?, CAST(? AS TIMESTAMP(6) WITH TIME ZONE), ?)",
    list(tstz_df.itertuples(index=False, name=None)),
)
con.commit()
for table_name in ["input_timeseries_compress_ts", "input_timeseries_compress_tstz"]:
    cur.execute(f"SELECT COUNT(*) FROM {table_name}")
    print(table_name, cur.fetchone()[0])

## 5. Configure Package SQL Builders

The table operator is installed once by `00_install_ts_compress_to_operator.ipynb`. This demo uses the package to generate SQL for both `TIMESTAMP(6)` and `TIMESTAMP(6) WITH TIME ZONE` inputs; only the input table and time column change.


In [ ]:
def make_timestamp_compressor(time_column: str) -> TeradataTimeSeriesCompressor:
    return TeradataTimeSeriesCompressor(
        CompressorSpec(
            id_columns=["series_id_1", "series_id_2"],
            time_column=time_column,
            value_column="y",
            time_type="timestamp",
            time_unit="second",
        ),
        operator_database=OPERATOR_DATABASE,
    )

print("Using mapping:", f"{OPERATOR_DATABASE + '.' if OPERATOR_DATABASE else ''}ts_compress_seq_to_map")


## 6. Run Compression and Measure Duration

This notebook demonstrates the two main ways to use the same compression operator on both `TIMESTAMP(6)` and `TIMESTAMP(6) WITH TIME ZONE` inputs:

- **Visual-fidelity compression**: `distance_max('1.0')`, `n_max('-1')`, and automatic tolerances calibrated on a small 100x100 plot space so sub-pixel noise is ignored.
- **Analytical point extraction**: `n_max('auto')`, `distance_max('-1')`, and `auto_log_drop('1.5')`.

Timestamp and date-like axes are converted internally to elapsed `time_unit` values from the first point in each partition. The generated signals include a small, sub-pixel amount of measurement noise to exercise the compression geometry without overwhelming it.

### Feature Engineering Note

The compression operator does not need to change to support feature engineering. After the operator emits selected rows, SQL can aggregate the result by series to compute retained point count, compression ratio, reduction percentage, and summaries of `compression_distance`. Those features can be joined back to a series catalog and used for clustering or monitoring: highly compressible series tend to look smooth or piecewise simple, while series that retain many points or keep a larger residual distance are usually more oscillatory, spiky, or noisy.

In [ ]:
mode_specs = {
    "visual_fidelity": visual_fidelity(width_px=100, height_px=100),
    "analytical_points": analytical_points(auto_log_drop=1.5).with_overrides(
        plot_width_px=100,
        plot_height_px=100,
    ),
}

source_columns_by_time = {
    "ts": ["series_id_1", "series_id_2", "ts", "y"],
    "ts_tz": ["series_id_1", "series_id_2", "ts_tz", "y"],
}

def build_timestamp_operator_sql(
    mode_name: str,
    source_table: str,
    time_column: str,
    parameters,
) -> str:
    compressor = make_timestamp_compressor(time_column)
    base_sql = compressor.compress(
        table_name=source_table,
        parameters=parameters,
        columns=source_columns_by_time[time_column],
        as_dataframe=False,
    )
    return f"""
SELECT
    '{mode_name}' AS compression_mode,
    t.*
FROM (
{base_sql}
) AS t
"""

timestamp_results = {}
for table_label, source_table, time_column, sort_column in [
    ("ts", "input_timeseries_compress_ts", "ts", "ts"),
    ("tstz", "input_timeseries_compress_tstz", "ts_tz", "ts_tz"),
]:
    for mode_name, parameters in mode_specs.items():
        operator_sql = build_timestamp_operator_sql(
            mode_name,
            source_table,
            time_column,
            parameters,
        )
        start = time.perf_counter()
        cur.execute(operator_sql + f" ORDER BY series_id_1, series_id_2, {sort_column}")
        rows = cur.fetchall()
        elapsed = time.perf_counter() - start
        key = f"{table_label}_{mode_name}"
        timestamp_results[key] = pd.DataFrame(
            rows,
            columns=[
                "compression_mode",
                "series_id_1",
                "series_id_2",
                time_column,
                "y",
                "compression_importance",
                "compression_distance",
            ],
        )
        denominator = len(ts_df) if table_label == "ts" else len(tstz_df)
        print(
            f"{key}: {len(timestamp_results[key]):,} / {denominator:,} rows; "
            f"duration: {elapsed:.3f} seconds"
        )

visual_fidelity_ts_df = timestamp_results["ts_visual_fidelity"]
analytical_points_ts_df = timestamp_results["ts_analytical_points"]
visual_fidelity_tstz_df = timestamp_results["tstz_visual_fidelity"]
analytical_points_tstz_df = timestamp_results["tstz_analytical_points"]

def summarize_points(df: pd.DataFrame, count_name: str) -> pd.DataFrame:
    return df.groupby(["series_id_1", "series_id_2"]).size().rename(count_name).reset_index()

print("TIMESTAMP visual-fidelity compression sample")
display(visual_fidelity_ts_df.head(5))
display(summarize_points(visual_fidelity_ts_df, "visual_points").head(8))

print("TIMESTAMP analytical point extraction sample")
display(analytical_points_ts_df.head(5))
display(summarize_points(analytical_points_ts_df, "analytical_points").head(8))

print("TIMESTAMP WITH TIME ZONE visual-fidelity compression sample")
display(visual_fidelity_tstz_df.head(5))
display(summarize_points(visual_fidelity_tstz_df, "visual_points").head(8))

print("TIMESTAMP WITH TIME ZONE analytical point extraction sample")
display(analytical_points_tstz_df.head(5))
display(summarize_points(analytical_points_tstz_df, "analytical_points").head(8))


## 7. Validate and Visualize Compression

In [ ]:
for name, compressed_df, original_len in [
    ("timestamp visual fidelity", visual_fidelity_ts_df, len(ts_df)),
    ("timestamp analytical points", analytical_points_ts_df, len(ts_df)),
    ("timestamp with time zone visual fidelity", visual_fidelity_tstz_df, len(tstz_df)),
    ("timestamp with time zone analytical points", analytical_points_tstz_df, len(tstz_df)),
]:
    assert len(compressed_df) < original_len, f"{name} should return a subset"
    assert compressed_df.groupby(["series_id_1", "series_id_2"]).size().min() >= 2
print("Validation passed for both timestamp tables and both use cases.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for ax, series_id_2 in zip(axes.ravel(), [1, 2, 3, 4]):
    original = ts_df[ts_df["series_id_2"] == series_id_2].copy()
    visual_fidelity = visual_fidelity_ts_df[visual_fidelity_ts_df["series_id_2"] == series_id_2].copy()
    analytical_points = analytical_points_ts_df[analytical_points_ts_df["series_id_2"] == series_id_2].copy()
    original["ts"] = pd.to_datetime(original["ts"])
    visual_fidelity["ts"] = pd.to_datetime(visual_fidelity["ts"])
    analytical_points["ts"] = pd.to_datetime(analytical_points["ts"])
    ax.plot(original["ts"], original["y"], linewidth=0.8, alpha=0.45, label="original")
    ax.plot(visual_fidelity["ts"], visual_fidelity["y"], linewidth=1.1, marker="o", markersize=2.6, label=f"visual fidelity ({len(visual_fidelity)})")
    ax.plot(analytical_points["ts"], analytical_points["y"], linewidth=1.3, marker="s", markersize=2.6, label=f"analytical points ({len(analytical_points)})")
    ax.set_title(f"{original['series_id_1'].iloc[0]} / series {series_id_2}")
    ax.grid(alpha=0.25)
    ax.legend(loc="best", fontsize=8)
fig.suptitle("TIMESTAMP visual-fidelity compression and analytical point extraction", y=1.02)
fig.tight_layout()
figure_path = FIGURE_DIR / "04_timestamp_compression_use_cases.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(figure_path)))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for ax, series_id_2 in zip(axes.ravel(), [1, 2, 3, 4]):
    original = tstz_df[tstz_df["series_id_2"] == series_id_2].copy()
    visual_fidelity = visual_fidelity_tstz_df[visual_fidelity_tstz_df["series_id_2"] == series_id_2].copy()
    analytical_points = analytical_points_tstz_df[analytical_points_tstz_df["series_id_2"] == series_id_2].copy()
    original["ts_tz"] = pd.to_datetime(original["ts_tz"], utc=True)
    visual_fidelity["ts_tz"] = pd.to_datetime(visual_fidelity["ts_tz"], utc=True)
    analytical_points["ts_tz"] = pd.to_datetime(analytical_points["ts_tz"], utc=True)
    ax.plot(original["ts_tz"], original["y"], linewidth=0.8, alpha=0.45, label="original")
    ax.plot(visual_fidelity["ts_tz"], visual_fidelity["y"], linewidth=1.1, marker="o", markersize=2.6, label=f"visual fidelity ({len(visual_fidelity)})")
    ax.plot(analytical_points["ts_tz"], analytical_points["y"], linewidth=1.3, marker="s", markersize=2.6, label=f"analytical points ({len(analytical_points)})")
    ax.set_title(f"{original['series_id_1'].iloc[0]} / series {series_id_2}")
    ax.grid(alpha=0.25)
    ax.legend(loc="best", fontsize=8)
fig.suptitle("TIMESTAMP WITH TIME ZONE visual-fidelity compression and analytical point extraction", y=1.02)
fig.tight_layout()
figure_path = FIGURE_DIR / "04_tstz_compression_use_cases.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(figure_path)))

## 8. Cleanup

In [ ]:
cur.close()
con.close()
print("Connection closed")